# Lab 2 — Text-to-SQL Security (Solutions)

Lab 1 produced a working text-to-SQL pipeline. This lab makes it **safe to ship**.

We'll work in layers, from cheap to expensive:

1. **Threat model.** What can go wrong, and which layer should stop each thing.
2. **Output guardrails.** SELECT-only, table allow-list, row cap, statement count.
3. **Execution guardrails.** Read-only connection, statement timeout.
4. **Data isolation.** A per-user view pattern that makes it *physically impossible*
   for one user's question to read another user's rows.
5. **Defence in depth.** Combine everything and attack it.

## Prerequisites

- You completed Lab 1, or you are happy reading the reference solution.
- `shop.db` exists next to this notebook (`python seed_db.py`).
- `OPENAI_API_KEY` is set.

## 1. Setup

In [ ]:
import os
import sqlite3
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
import sqlglot
from sqlglot import exp

load_dotenv()
DB_PATH = Path("shop.db")
MODEL = "gpt-4o-mini"
client = OpenAI()

## 2. Threat model

Take five minutes and ask: *if a malicious user can type anything into the question
box, what bad outcomes are possible?* Below is a non-exhaustive starter list. The
rest of the lab is organized around defending each of them.

| # | Threat                                                       | Stopped by                                              |
|---|--------------------------------------------------------------|---------------------------------------------------------|
| 1 | "Delete all customers" — destructive DML/DDL                 | SELECT-only AST check                                   |
| 2 | "What's in `users_secret`?" — read-anything                  | Table allow-list                                        |
| 3 | Query that scans a 10M-row table for hours                   | Row cap + statement timeout                             |
| 4 | Multi-statement payload: `SELECT 1; DROP TABLE customers;`   | Reject multiple top-level statements                    |
| 5 | "Show me Alice's orders" (asked by Bob)                      | Per-user view / row-level isolation                     |
| 6 | Prompt injection that smuggles a `UNION` to other tables     | Static SELECT scope + isolation (a malicious SQL still cannot reach outside the user's view) |

Threats 1–4 are **output guardrails** (what the SQL is allowed to *say*).
Threat 5 is **data isolation** (what the SQL is allowed to *see*, regardless of what
it says).
Threat 6 is a combination — neither layer alone is enough; both are.

## 3. Output guardrails

Re-introduce the validators from Lab 1, plus two new checks:

- **Single statement.** `sqlglot.parse` returns a list; reject inputs with more than one.
- **Enforced LIMIT.** If the query is unbounded, inject `LIMIT n` rather than failing
  (better UX than rejecting an otherwise valid question).

In [ ]:
ALLOWED_TABLES = {"customers", "products", "orders", "order_items", "reviews"}
MAX_ROWS = 100
FORBIDDEN = (exp.Insert, exp.Update, exp.Delete, exp.Drop, exp.Create, exp.AlterTable, exp.TruncateTable)


def parse_one_or_raise(sql: str) -> exp.Expression:
    """Reject inputs that contain more than one top-level statement."""
    statements = [s for s in sqlglot.parse(sql, dialect="sqlite") if s is not None]
    if len(statements) != 1:
        raise ValueError(f"expected exactly one statement, got {len(statements)}")
    return statements[0]

### TODO 3.1 — `enforce_limit`

In [ ]:
def enforce_limit(sql: str, max_rows: int = MAX_ROWS) -> str:
    tree = parse_one_or_raise(sql)
    existing = tree.args.get("limit")
    if existing is None:
        tree = tree.limit(max_rows)
    else:
        try:
            current = int(existing.expression.this)
        except Exception:
            current = max_rows + 1
        if current > max_rows:
            tree = tree.limit(max_rows)
    return tree.sql(dialect="sqlite")


print(enforce_limit("SELECT name FROM customers"))
print(enforce_limit("SELECT name FROM customers LIMIT 5"))
print(enforce_limit("SELECT name FROM customers LIMIT 10000"))

### TODO 3.2 — `validate_sql_strict`

In [ ]:
def validate_sql_strict(sql: str, allowed: set[str] = ALLOWED_TABLES) -> tuple[bool, str]:
    try:
        tree = parse_one_or_raise(sql)
    except Exception as e:
        return False, f"parse_error: {e}"
    if not isinstance(tree, exp.Select):
        return False, "not_select"
    if any(isinstance(n, FORBIDDEN) for n in tree.walk()):
        return False, "forbidden_node"
    tables = {t.name for t in tree.find_all(exp.Table)}
    bad = tables - allowed
    if bad:
        return False, f"unknown_tables: {sorted(bad)}"
    return True, "ok"


for sample in [
    "SELECT * FROM customers",
    "SELECT 1; DROP TABLE customers",
    "DELETE FROM customers",
    "SELECT * FROM secret_table",
    "WITH x AS (DELETE FROM customers RETURNING *) SELECT * FROM x",
]:
    print(sample, "->", validate_sql_strict(sample))

## 4. Execution guardrails

Even a perfectly validated `SELECT` can still hurt you:

- A cartesian join over five tables.
- A regex over a million `comment` strings.
- A `LIKE '%...%'` triggering a full scan.

Two cheap defences at the connection layer:

1. **Read-only mode** — `sqlite3.connect("file:shop.db?mode=ro", uri=True)`. Bullet-
   proof: even a SELECT-only AST that mutated the DB via a SQLite extension would be
   blocked here.
2. **Statement timeout** — SQLite exposes `set_progress_handler`. We register a
   callback that raises after N "ticks", which is roughly proportional to how long
   the query has been running.

In [ ]:
class QueryTimeout(Exception):
    pass


def execute_safely(sql: str, max_rows: int = MAX_ROWS, timeout_ticks: int = 1_000_000) -> pd.DataFrame:
    """Read-only execution with a soft timeout and a hard row cap.

    `timeout_ticks` is *not* wall-clock time — it is the number of SQLite VM operations
    between progress callbacks. 1,000,000 is roughly a second on modern hardware.
    For real systems use a wall-clock timeout via a thread or subprocess.
    """
    uri = f"file:{DB_PATH.as_posix()}?mode=ro"
    ticks = {"n": 0}

    def handler():
        ticks["n"] += 1
        if ticks["n"] > 100:
            raise QueryTimeout("timeout")
        return 0  # 0 = continue, non-zero = abort

    with sqlite3.connect(uri, uri=True) as conn:
        conn.set_progress_handler(handler, timeout_ticks // 100)
        df = pd.read_sql_query(sql, conn)
    return df.head(max_rows)


# Happy path
print(execute_safely("SELECT name FROM customers LIMIT 5"))

## 5. Data isolation — the per-user view pattern

Output guardrails answer *what may the query do?*. They do **not** answer *which rows may
the query see?*. For that, you need to make the rows the user shouldn't see invisible
at the connection level — so even a perfectly valid SQL statement cannot reach them.

Postgres has first-class **row-level security** (RLS) policies. SQLite doesn't. The
portable pattern (also used in many SaaS deployments behind Postgres) is:

> For each logged-in user, expose a **view** that is filtered to only their rows, and
> give them a database role that can only read that view.

Concretely:

- Create a per-request **scratch database** that contains views like
  `my_orders`, `my_order_items`, `my_reviews`, filtered by `customer_id = :uid`.
- Generate SQL against the scratch schema, not the global one.
- The model never sees the real `customer_id` column and can never join its way to
  another user's rows.

We'll implement this with SQLite's `ATTACH DATABASE` + `CREATE TEMP VIEW`, which is
small enough to inspect end-to-end.

### TODO 5.1 — `open_user_session`

In [ ]:
def open_user_session(customer_id: int) -> sqlite3.Connection:
    if not isinstance(customer_id, int):
        raise TypeError("customer_id must be int")
    uri = f"file:{DB_PATH.as_posix()}?mode=ro"
    conn = sqlite3.connect(uri, uri=True)
    # We embed the integer ID directly — but only after type-checking it above.
    # Parameter binding doesn't work inside CREATE VIEW bodies on SQLite.
    cur = conn.cursor()
    cur.execute(f"CREATE TEMP VIEW my_profile AS SELECT customer_id, name, email, country, created_at FROM customers WHERE customer_id = {customer_id}")
    cur.execute(f"CREATE TEMP VIEW my_orders AS SELECT order_id, order_date, status, total_amount FROM orders WHERE customer_id = {customer_id}")
    cur.execute(f"CREATE TEMP VIEW my_order_items AS "
                f"SELECT oi.order_item_id, oi.order_id, oi.product_id, oi.quantity, oi.unit_price "
                f"FROM order_items oi JOIN orders o ON o.order_id = oi.order_id "
                f"WHERE o.customer_id = {customer_id}")
    cur.execute(f"CREATE TEMP VIEW my_reviews AS SELECT review_id, product_id, rating, comment, review_date FROM reviews WHERE customer_id = {customer_id}")
    return conn


conn = open_user_session(1)
print(pd.read_sql_query("SELECT * FROM my_profile", conn))
print(pd.read_sql_query("SELECT order_id, status, total_amount FROM my_orders LIMIT 3", conn))
conn.close()

### TODO 5.2 — text-to-SQL against the user view

Build a translator that targets the *user-scoped schema*, not the real one. The model
is told only about `my_profile`, `my_orders`, `my_order_items`, `my_reviews`, plus the
read-only `products` catalogue.

Notice what this buys you: even if the model wrote `SELECT * FROM my_orders`, it
physically cannot see another customer's orders, because the view filters them out.

In [ ]:
USER_SCHEMA_DDL = """
-- catalogue (shared, read-only)
CREATE TABLE products (
    product_id INTEGER PRIMARY KEY, name TEXT, category TEXT, price REAL, stock INTEGER
);
-- the current user's data
CREATE VIEW my_profile     (customer_id, name, email, country, created_at);
CREATE VIEW my_orders      (order_id, order_date, status, total_amount);
CREATE VIEW my_order_items (order_item_id, order_id, product_id, quantity, unit_price);
CREATE VIEW my_reviews     (review_id, product_id, rating, comment, review_date);
"""

ALLOWED_USER_TABLES = {"products", "my_profile", "my_orders", "my_order_items", "my_reviews"}


def translate_user_sql(question: str) -> str:
    system = (
        "You write SQLite SQL queries that answer questions about the current user's data.\n"
        "You may ONLY reference these objects: products, my_profile, my_orders, my_order_items, my_reviews.\n"
        "Do NOT reference customers, orders, order_items, or reviews directly.\n"
        "SQL only, no fences, no commentary.\n\nSchema:\n" + USER_SCHEMA_DDL
    )
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": question},
        ],
        temperature=0,
    )
    return resp.choices[0].message.content.strip().strip("`")


sql = translate_user_sql("How much did I spend on Electronics in total?")
print(sql)

### Putting the user-scoped pipeline together

Below we wrap everything: open a session for user 1, translate, validate against the
user-scoped allow-list, then execute against the user's connection. Note that
`execute_safely` accepts a connection argument — see the cell.

In [ ]:
def validate_user_sql(sql: str) -> tuple[bool, str]:
    try:
        tree = parse_one_or_raise(sql)
    except Exception as e:
        return False, f"parse_error: {e}"
    if not isinstance(tree, exp.Select):
        return False, "not_select"
    if any(isinstance(n, FORBIDDEN) for n in tree.walk()):
        return False, "forbidden_node"
    tables = {t.name for t in tree.find_all(exp.Table)}
    bad = tables - ALLOWED_USER_TABLES
    if bad:
        return False, f"forbidden_tables: {sorted(bad)}"
    return True, "ok"


def ask_as_user(customer_id: int, question: str) -> pd.DataFrame:
    sql = translate_user_sql(question)
    ok, why = validate_user_sql(sql)
    print(f"[user {customer_id}] SQL: {sql}")
    print(f"[user {customer_id}] validation: {ok} ({why})")
    if not ok:
        raise PermissionError(why)
    conn = open_user_session(customer_id)
    try:
        return pd.read_sql_query(sql, conn).head(MAX_ROWS)
    finally:
        conn.close()


ask_as_user(1, "What is the total amount of all my orders?")

## 6. Demonstrating isolation — three attacks, three defences

Below are three attempted misuses. After each one, take a moment to ask yourself
*which layer caught this?*

### 6.1 — Destructive command

In [ ]:
# The model is told never to mutate; even if a user begs, validate_sql_strict catches it.
bad_sql = "DELETE FROM customers"
print(validate_sql_strict(bad_sql))

### 6.2 — Reading another user's data, the obvious way

In [ ]:
# User 1 asks about user 2's orders.
attempt = "SELECT order_id, total_amount FROM orders WHERE customer_id = 2"
# Against the user-scoped allow-list this is rejected — the table 'orders' isn't allowed.
print(validate_user_sql(attempt))

### 6.3 — Reading another user's data, the sneaky way

Suppose the model is somehow tricked (prompt injection, hostile question) into writing
a query that *only* uses allowed objects but tries to reach a different customer's data
through them. With the user view in place, the row filter is already applied — the
rows are not in the result set, regardless of the WHERE clause.

In [ ]:
# Even this passes validation (it only references my_orders), but the view physically
# has no rows for customer_id != current user, so the WHERE clause matches nothing.
sneaky = "SELECT order_id, total_amount FROM my_orders WHERE order_id IN (SELECT order_id FROM my_orders)"
ok, why = validate_user_sql(sneaky)
print("validation:", ok, why)

# Open a session as user 1, then check: are there any rows where the row actually
# belongs to user 2? There shouldn't be.
conn = open_user_session(1)
leak_check = pd.read_sql_query(
    """SELECT COUNT(*) AS leaked_rows
       FROM my_orders mo
       JOIN main.orders o ON o.order_id = mo.order_id
       WHERE o.customer_id != 1""",
    conn,
)
print(leak_check)
conn.close()

> Note: that leak-check query had to reach into `main.orders` (the real table) to even
> pose the question. In a production setting you would not expose that table on the
> user-scoped connection at all — for example by attaching `shop.db` under a sealed
> schema and only granting access to the views. SQLite's permission model is weak;
> Postgres's `GRANT` system makes this enforcement first-class.

## 7. Defence in depth — end-to-end run

One small interactive demo. We loop over a few questions and run them as two different
users; you should see the same question return different rows depending on who asks.

In [ ]:
questions = [
    "How many orders have I placed?",
    "Show me the products I have reviewed and my rating.",
    "What categories have I spent the most on?",
]

for uid in (1, 2):
    print(f"\n=== Asking as customer_id = {uid} ===")
    for q in questions:
        print(f"\nQ: {q}")
        try:
            df = ask_as_user(uid, q)
            print(df.to_string(index=False))
        except Exception as e:
            print("blocked:", e)

## Summary

You built three layers of defence and saw each one stop a different attack class:

1. **Output guardrails** (AST checks, allow-list, single-statement, enforced LIMIT)
   — cheap, deterministic, applied before execution.
2. **Execution guardrails** (read-only connection, soft timeout, row cap) — defend
   against ill-behaved-but-legal queries.
3. **Per-user views** — the only layer that defends against the model *correctly*
   writing a query against the wrong data. Without this, a sufficiently clever
   prompt-injection could read across users.

None of these layers is sufficient alone. Each one is cheap; together they are very
hard to get around.

## Exercises

1. **PII redaction.** Add a step that masks any `email` column in the response before
   returning it. The model must still be able to write queries referencing emails for
   filtering, but the values must never reach the user.
2. **Audit log.** Append every translation + execution to a `query_log` table
   (separate database). Include the question, generated SQL, validation result, row
   count, and a hash of the user ID.
3. **Adversarial test set.** Build a list of 10 hostile prompts (DML, cross-user
   reads, prompt injections) and assert that all of them are rejected. Add this as a
   regression test.
4. **Postgres port.** Sketch how you would replace the temp-view trick with native
   Postgres RLS policies (`CREATE POLICY ... USING (customer_id = current_setting('app.user_id')::int)`).
   What is gained, what is lost?